In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()
# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

###########################
# Import CRUD Python Module
###########################
import sys
sys.path.insert(0, '/home/codio/workspace/code_files')
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Initialize CRUD object with aacuser credentials
username = "aacuser"
password = "MySecurePassword123"
shelter = AnimalShelter(username, password)

# Use read method to retrieve all documents from MongoDB
# Empty dictionary {} returns all documents
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB returns '_id' column with ObjectID type - remove it to prevent errors in data_table
df.drop(columns=['_id'], inplace=True)

## Debug - uncomment to verify data loaded
# print(f"Total records: {len(df.to_dict(orient='records'))}")
# print(f"Columns: {df.columns.tolist()}")

#########################
# Dashboard Layout / View
#########################
app = JupyterDash('AnimalShelterDashboard')

app.layout = html.Div([
    # Hidden div for inter-component communication
    html.Div(id='hidden-div', style={'display':'none'}),
    
    # Header with unique identifier
    html.Center(html.B(html.H1('Grazioso Salvare - Animal Shelter Dashboard'))),
    html.Center(html.H3('Austin Animal Center Outcomes Database')),
    html.Hr(),
    
    # Interactive Data Table
    html.H2('Animal Shelter Records'),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Features for user-friendly interactive data table
        page_size=10,  # Show 10 rows per page
        style_header={'backgroundColor': 'rgb(30, 30, 30)', 'color': 'white', 'fontWeight': 'bold'},
        style_cell={'textAlign': 'left', 'padding': '10px'},
        row_selectable='single',  # Allow single row selection only
        selected_rows=[0],  # Select first row by default
        filter_action="native",  # Enable filtering
        sort_action="native",  # Enable sorting
        page_action="native"  # Enable pagination
    ),
    
    html.Br(),
    html.Hr(),
    
    # Geolocation Map
    html.H2('Animal Location Map'),
    html.Div(
        id='map-id',
        className='col s12 m6',
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback to highlight selected columns in data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """
    Highlight selected columns with light blue background
    """
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Callback to update geolocation chart when row is selected
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    """
    Update map based on selected row in data table
    
    Args:
        viewData: The data visible in the filtered/sorted table
        index: List of selected row indices (single selection only)
    """
    # Convert the view data to a dataframe
    dff = pd.DataFrame.from_dict(viewData)
    
    # Handle case where no row is selected - default to first row
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
    
    # Austin TX center coordinates [latitude, longitude]
    # Column 13 = latitude, Column 14 = longitude (based on AAC dataset structure)
    # Column 4 = breed, Column 9 = animal name
    
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],  # Austin, TX coordinates
            zoom=10,
            children=[
                # Base map layer
                dl.TileLayer(id="base-layer-id"),
                # Marker for selected animal location
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        # Tooltip shows breed when hovering
                        dl.Tooltip(dff.iloc[row, 4]),
                        # Popup shows animal name when clicking marker
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode
# Note: if port 8050 is unavailable, try an alternate port
app.run_server()

Dash app running on https://gammainch-gopherspeech-3000.codio.io/proxy/8050/
